# ChaosAgentGym — Colab one-click trainer

**Before running anything:**
1. `Runtime → Change runtime type → T4 GPU → Save` (mandatory; T4 free tier works).
2. Run cells top-to-bottom. Each cell prints what it expects to do and the expected runtime.
3. The big training cell (~25 min) runs unattended; you can leave the tab open.

**What you'll need ready:** `chaos_agent_gym.zip` from your laptop (the file your assistant generated alongside this notebook).

**Total wall-clock:** ~30 min on a T4 (5 min SFT warmup + 25 min RL).

## 0. GPU sanity
Should print a Tesla T4 (or better). If it errors, GPU isn't enabled — fix the runtime setting first.

In [ ]:
!nvidia-smi

## 1. Upload the project
Browse-pick `chaos_agent_gym.zip` when the dialog appears. Skip this cell on re-runs of the same Colab session.

In [ ]:
import os
if not os.path.isdir('/content/chaos_agent_gym'):
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !unzip -q "{zip_name}" -d /content/
    # The local folder is 'chaos agent gym' (with spaces) — rename for sanity.
    if os.path.isdir('/content/chaos agent gym'):
        !mv "/content/chaos agent gym" /content/chaos_agent_gym
%cd /content/chaos_agent_gym
!ls

## 2. Install dependencies
First-run takes ~2 min. Idempotent on re-runs.

In [ ]:
!pip -q install 'torch>=2.1' 'transformers>=4.44' 'trl>=0.8.6,<0.13' 'accelerate>=0.33' matplotlib

## 3. Env sanity (no GPU needed)
Expected: scripted oracle ~78% on the single task, ~77% on the mixed curriculum. If these don't match, something broke during upload.

In [ ]:
!python -m env.smoke_test
print('---')
!python -m env.test_tasks
print('---')
!python -m training.test_pipeline

## 4. Establish baselines BEFORE training
This is the "before" half of your demo — keep this output handy.

In [ ]:
!python -m eval.quantitative --n_seeds 100 --task_curriculum

## 5. Generate SFT demos
~5 seconds. Produces `logs/demos.jsonl` with both task variants.

In [ ]:
!python -m training.make_demo_dataset --n_episodes 300 --task_curriculum

## 6. SFT warmup (~5–7 min on T4)
**T4 memory fix:** the first sub-cell patches `sft_warmup.py` to enable gradient checkpointing (cuts activation memory by ~50%). The second runs warmup with smaller batch + shorter context.

Loss should drop from ~3 to ~0.5 by end of epoch.

In [ ]:
# Patch sft_warmup.py to enable gradient checkpointing (idempotent)
src_path = '/content/chaos_agent_gym/training/sft_warmup.py'
src = open(src_path).read()
if 'gradient_checkpointing_enable' not in src:
    src = src.replace(
        'model.to(device)\n    model.train()',
        'model.to(device)\n'
        '    model.gradient_checkpointing_enable()\n'
        '    model.config.use_cache = False\n'
        '    model.train()'
    )
    open(src_path, 'w').write(src)
    print('patched sft_warmup.py — gradient checkpointing enabled')
else:
    print('sft_warmup.py already patched')

!python -m training.sft_warmup \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --demos logs/demos.jsonl \
    --epochs 1 --lr 2e-5 \
    --batch 2 --max_len 768 \
    --out logs/ckpt_sft

## 7. RL training — TRL path (~25 min on T4)
First sub-cell patches `train_trl.py` for the same memory fix. Then runs PPO.

Watch `mean_return` per update. Healthy curve: starts near 0, climbs toward +0.5 over 25 updates.

If this errors with a TRL version mismatch, **skip it** and run cell 7b below.

In [ ]:
# Patch train_trl.py to enable gradient checkpointing on the value-head model
src_path = '/content/chaos_agent_gym/training/train_trl.py'
src = open(src_path).read()
if 'gradient_checkpointing_enable' not in src:
    src = src.replace(
        'ref_model.to(device)',
        'ref_model.to(device)\n'
        '    model.pretrained_model.gradient_checkpointing_enable()\n'
        '    model.pretrained_model.config.use_cache = False'
    )
    open(src_path, 'w').write(src)
    print('patched train_trl.py')
else:
    print('train_trl.py already patched')

!python -m training.train_trl \
    --model logs/ckpt_sft \
    --episodes 200 --batch 4 --mini_batch 2 --lr 1e-6 \
    --task_curriculum --difficulty_ramp

### 7b. RL fallback — hand-rolled REINFORCE (only run if cell 7 errored)
Same outputs to `logs/`, no TRL dependency. Also patched for memory.

In [ ]:
# Patch train.py for memory then run
src_path = '/content/chaos_agent_gym/training/train.py'
src = open(src_path).read()
if 'gradient_checkpointing_enable' not in src:
    src = src.replace(
        'model.train()',
        'model.gradient_checkpointing_enable()\n'
        '    model.config.use_cache = False\n'
        '    model.train()',
        1  # only first occurrence
    )
    open(src_path, 'w').write(src)
    print('patched train.py')
else:
    print('train.py already patched')

# Uncomment the next block to run the fallback trainer
# !python -m training.train \
#     --model logs/ckpt_sft \
#     --episodes 200 --batch 4 --lr 1e-6 \
#     --task_curriculum --difficulty_ramp

## 8. Plot the reward curve

In [ ]:
!python -m eval.plot_rewards --csv logs/rewards.csv --out logs/reward_curve.png
from IPython.display import Image
Image('logs/reward_curve.png')

## 9. Quantitative eval — base vs trained
Held-out 100 seeds, disjoint from training. The `success` column for the trained row should beat the base row by a wide margin.

In [ ]:
!python -m eval.quantitative \
    --base Qwen/Qwen2.5-0.5B-Instruct \
    --trained logs/ckpt_final \
    --n_seeds 100 --task_curriculum

## 10. Behavioral fingerprint — *how* did behavior change?
The headline numbers for the demo:
- premature-VERIFY % should drop
- defended-VERIFY % (≥2 PUTs) should rise
- recovery-after-failure % should rise

In [ ]:
!python -m eval.behavior_diff \
    --base Qwen/Qwen2.5-0.5B-Instruct \
    --trained logs/ckpt_final \
    --n_seeds 50

## 11. Side-by-side transcripts — base vs trained on identical seeds

In [ ]:
!python -m eval.before_after \
    --base Qwen/Qwen2.5-0.5B-Instruct \
    --trained logs/ckpt_final \
    --seeds 7 13 21 34 55 89
from IPython.display import Markdown
Markdown(open('logs/before_after.md').read())

## 12. Download all artifacts to your laptop
Bundles the curve, transcripts, eval tables, and the trained checkpoint into `results.zip`.

In [ ]:
!zip -rq results.zip \
    logs/reward_curve.png \
    logs/rewards.csv \
    logs/before_after.md \
    logs/quantitative.md logs/quantitative.json \
    logs/behavior_diff.md \
    logs/transcripts.txt \
    logs/ckpt_final
from google.colab import files
files.download('results.zip')